
### ReCoのTcデータを用いたクロスバリデーションによるKernelRidge回帰

**データ取得からデータ解析**

ReCoのTcデータを用いてKernelRidge回帰を行う。

CVによりモデル性能評価をおこない、ハイパーパラメタを決める。


In [ ]:
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pandas as pd
import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 10)

In [ ]:
# "データ取得
def get_data(data_name: str):
    """データ取得

    Args:
        data_name (str): データ名
    """
    if data_name=="ReCo":
        df = pd.read_csv("../data/TC_ReCo_detail_descriptor.csv")
        descriptor_names = ['C_R', 'C_T', 'vol_per_atom', 'Z', 'f4', 'd5', 'L4f', 'S4f', 'J4f',
            '(g-1)J4f', '(2-g)J4f']
        # descriptor_names = ['C_R', 'vol_per_atom','Z']
        target_name = 'Tc'
    elif data_name=="ZBWE":
        df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
        descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
                            'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
                            'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
        target_name = 'dE'

    return df, descriptor_names, target_name

g_data_name = "ReCo"
g_df, g_descriptor_names, g_target_name = get_data(g_data_name)

In [ ]:
# 結果を入れるdict
g_result = {}

In [ ]:
g_Xraw = g_df[g_descriptor_names].values
g_y = g_df[g_target_name].values

# データプリプロセス
g_scaler = MinMaxScaler()
g_scaler.fit(g_Xraw)
g_X = g_scaler.transform(g_Xraw)

# データ解析
nfold = 5
g_kf = KFold(nfold, shuffle=True)
#kf = LeaveOneOut()
g_estimator = KernelRidge(alpha=1, gamma=1, kernel="rbf")
g_kr = GridSearchCV(g_estimator, scoring="r2",
                  cv=g_kf, param_grid={"alpha": np.logspace(-6, 0, 11), 
                                       "gamma": np.logspace(-4, 0, 10)})
g_kr.fit(g_X, g_y)
g_kr_score = g_kr.score(g_X, g_y)
print("R2=", g_kr_score)
g_result["GridSearchCV"] = {"R2": g_kr_score}

g_yp = g_kr.predict(g_X)

**可視化**

In [ ]:
g_df

In [ ]:
def show_X(X):
    """Xの図示

    Args:
        X (np.ndarray): 説明変数
    """
    fig, ax = plt.subplots()
    plt.plot(X, ".-")
    plt.show()
show_X(g_X)

def show_hist(y):
    """yの図示

    Args:
        y (np.ndarray): 目的変数
    """
    fig, ax = plt.subplots()
    ax.hist(y)
    ax.set_xlabel("y")
    fig.show()
show_hist(g_y)

GridSearchCV得られた予測値を表示します。

In [ ]:
#krbest = kr.best_estimator_
g_krbest = g_kr
g_yp = g_krbest.predict(g_X)

def show_y_yp(y,yp):
    """y vs ypの図示。

    Args:
        y (np.ndarray): 目的変数観測値
        yp (np.ndarray): 目的変数予測値
    """
    fig, ax =plt.subplots(figsize=(5, 5))
    ax.plot(y, yp, "o")
    yall = np.hstack([yp, y])
    y1, y2 = np.min(yall), np.max(yall)
    ax.plot([y1, y2], [y1, y2], "--")  # 対角線を引く
    ax.set_xlabel("$y_{obs}$")
    ax.set_ylabel("$y_{pred}$")
    fig.show()
    
show_y_yp(g_y,g_yp)



辞書kr.cv_results_にCVの詳細な結果が含まれています。これを見ていきます。

ハイパーパラメタの空間kr.cv_results_["param_alpha"]、kr.cv_results_["param_gamma"]
に対してkr.cv_results_["mean_test_score"]の値を見ます。
$R^2$の最大値が今のハイパーパラメタの移動範囲に存在するように見えます。

In [ ]:
# %matplotlib notebook
g_alpha_mesh = np.log10(
    g_kr.cv_results_["param_alpha"].data.astype(float)).reshape(11, 10)
g_gamma_mesh = np.log10(
    g_kr.cv_results_["param_gamma"].data.astype(float)).reshape(11, 10)
#std_test_score = kr.cv_results_["std_test_score"].reshape(11,10)
g_mean_test_score = g_kr.cv_results_["mean_test_score"].reshape(11, 10)
g_angle = 0
g_fig = plt.figure()
g_ax = g_fig.add_subplot(111, projection="3d")
g_ax.plot_surface(g_alpha_mesh, g_gamma_mesh, g_mean_test_score)
g_ax.set_xlabel("alpha")
g_ax.set_ylabel("gamma")
g_ax.set_zlabel("R2")
g_ax.set_title("angle={}".format(g_angle))
g_ax.view_init(30, g_angle)
# ax.set_zlim((0.8,1.1))
g_fig.show()

最適なハイパーパラメタの値とその時のindexを示します。

In [ ]:
print(g_kr.best_index_)
print(g_kr.best_params_)

kr.best_index_は下のnp.argmaxの値と同じです。最適なハイパーパラメタも同じ値です。

In [ ]:
g_iopt = np.argmax(g_kr.cv_results_["mean_test_score"])
print(g_iopt, "best_mean_test_score=", g_kr.cv_results_["mean_test_score"][g_iopt])
g_alpha_opt = g_kr.cv_results_["param_alpha"][g_iopt]
g_gamma_opt = g_kr.cv_results_["param_gamma"][g_iopt]
print("alpha,gamma(opt)=", g_alpha_opt, g_gamma_opt)


### 付録


最適なハイパーパラメタを用いてKernelRidgeモデルを作り直し、GridSearchCVの予測値と同じであることを示します。

In [ ]:
g_kropt = KernelRidge(alpha=g_alpha_opt, gamma=g_gamma_opt,kernel="rbf")
g_kropt.fit(g_X, g_y)
g_yp2 = g_kropt.predict(g_X)
print(g_yp-g_yp2) # 同じ値であることを示す。
g_score = g_kropt.score(g_X,g_y)
print("R2 score=", g_score)
g_result["KR_opt"] = {"R2": g_score} 

In [ ]:
#R2の比較
g_dfresult = pd.DataFrame(g_result)
g_dfresult["diff"] = g_dfresult["GridSearchCV"]- g_dfresult["KR_opt"]
g_dfresult

GridSearchCVも

1. CVで最も良いハイパーパラメタを得る。
2. 最も良いハイパーパラメタで全ての（X,y)を用いて回帰モデルを作りなおす。

ということをしています。

### $y^{obs}$ vs $y^{pred}_{CV(test)}$の表示
CV(test)の$R^2$と
目的変数の値 vs CV(test)の予測値の表示を行います。
しかし、krオブジェクトの中に入っていないので最も良いパラメタを用いて再計算を行います。

In [ ]:
from sklearn.metrics import r2_score

def make_CV_y_yp_test(X, y, alpha_opt, gamma_opt, nsplit=5):
    """CVによりy, ypを計算する。

    Args:
        X (np.ndarray): 説明変数。
        y (np.ndarray): 目的変数。
        alpha_opt (float):  alpha of RBF.
        gamma_opt (float):  gamma of RBF.
        nsplit (int, optional): KFoldの分割数. Defaults to 5.

    Returns:
        [np.ndarray]: a list of y.
        [np.ndarray]: a list of predicted y.
        [float]: a list of CV score.
    """
    yp_list = []
    y_list = []
    score_list = []
    kf = KFold(nsplit, shuffle=True)
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        krcv = KernelRidge(alpha=alpha_opt, gamma=gamma_opt, kernel="rbf")
        krcv.fit(Xtrain, ytrain)
        ytestp = krcv.predict(Xtest)
        score = r2_score(ytest, ytestp)
        score_list.append(score)
        y_list.extend(ytest)
        yp_list.extend(ytestp)
    return y_list, yp_list, score_list

g_y_list, g_yp_list, g_score_list = make_CV_y_yp_test(g_X, g_y, g_alpha_opt, g_gamma_opt,)

乱数によりクロスバリデーションの分割が異なるので以下のR2 CV(test)の値は少し異なります。

In [ ]:
print("R2 CV(test)={}({})".format(np.mean(g_score_list),np.std(g_score_list)))

In [ ]:
%matplotlib inline

show_y_yp(g_y_list, g_yp_list)


**補足**

* HSCI Lasso（Hilbert-Schmidt Independence Criterion Lasso）というカーネル法でL1正則化を行い特徴量選択を行う手法もあります。

ref. https://github.com/riken-aip/pyHSICLasso



* LASSOは回帰と同時に説明変数選択ができていましたが、
Kernel回帰の場合は説明変数を与えて回帰をするので説明変数選択の議論をしたい場合は別途作業が必要です。

ref. https://doi.org/10.7566/JPSJ.87.113801


##### 問題

hyperparameterを最適化した線形モデルのRMSEとの比較

他のデータでの実行